# Extract dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir /content/input

In [ ]:
!unzip -q /content/drive/MyDrive/Datasets/s2/no_stride/fase3/NDRE.zip -d /content/input/dataset_final

split  cidade                 
test   oliveira                   195
train  bom_sucesso                156
       candeias                   151
       nazareno                    87
       perdoes                     75
       sao_francisco_de_paula      73
val    camacho                     57
       santo_antonio_do_amparo    108
dtype: int64

Copiando 902 arquivos...
Processo concluído.


# S2-L2A Coffee Plantation Segmentation

    Aluno: Matheus Martins Batista

    Universidade Federal de Itajubá

In [ ]:
!python --version

Python 3.12.12


## Init Enviroment

### Install Packages

In [6]:
!pip install -q -U segmentation-models-pytorch timm albumentations rasterio torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 73.6 MB/s eta 0:00:00


### Import libs and def Global Envs

In [7]:
import os, random, gc, warnings, time, shutil
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio
import cv2
import matplotlib.pyplot as plt
import torchmetrics
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
from ipywidgets import interact, IntSlider

TIMESTAMP    = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")
SEED         = 115
DATA_ROOT    = Path('/content/input/dataset_final')
BASE_DIR = Path(f"/content/working_{TIMESTAMP}")
BATCH_SIZE   = 16
NUM_EPOCHS   = 100
LR           = 1e-4
THR          = 0.5
EARLY_STOP_PATIENCE = 15
SAMPLE_CHANNELS = 5

USE_NORMALIZATION = False
DATASET_MEAN = []
DATASET_STD  = []

HAS_VIS = True
VIS_CLIP_RANGE = (-1, 1)
IMAGE_GAIN = 5.5

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DIRS = {
    'train': {
        'img': DATA_ROOT / 'train' / 'images',
        'mask': DATA_ROOT / 'train' / 'masks'
    },
    'val': {
        'img': DATA_ROOT / 'val' / 'images',
        'mask': DATA_ROOT / 'val' / 'masks'
    },
    'test': {
        'img': DATA_ROOT / 'test' / 'images',
        'mask': DATA_ROOT / 'test' / 'masks'
    }
}

BASE_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
seed_everything()

warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

print(BASE_DIR)

/content/working_10_04_2026_03_34_41


### Utils

In [8]:
def view_sample(index):
    tensor_img  = batch_imgs[index]
    tensor_mask = batch_masks[index]

    img_np = tensor_img.permute(1, 2, 0).cpu().numpy()

    if 'USE_NORMALIZATION' in globals() and USE_NORMALIZATION:
        mean = np.array(DATASET_MEAN)
        std  = np.array(DATASET_STD)
        img_disp = (img_np * std) + mean
        title_prefix = "Denormalized"
    else:
        img_disp = img_np
        title_prefix = "Raw"

    rgb = img_disp[:, :, 0:3]
    rgb = np.clip(rgb * IMAGE_GAIN, 0.0, 1.0)

    msk = tensor_mask[0].cpu().numpy()

    has_vis = 'HAS_VIS' in globals() and HAS_VIS

    n_cols = 3 if has_vis else 2

    fig, ax = plt.subplots(1, n_cols, figsize=(6 * n_cols, 6))

    ax[0].imshow(rgb)
    ax[0].set_title(f"RGB {title_prefix}")
    ax[0].axis('off')

    ax[1].imshow(msk, cmap='gray')
    ax[1].set_title("Mask")
    ax[1].axis('off')

    if has_vis:
        vis = img_disp[:, :, -1]

        im = ax[2].imshow(vis, cmap='RdYlGn')
        ax[2].set_title("Vegetation Index (VI)")
        ax[2].axis('off')

        fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

def get_stats(file_list):
    ds = CoffeeDataset(file_list, augment=None, has_vis=HAS_VIS)
    loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)

    sample, _ = ds[0]
    n_channels = sample.shape[0]

    mean = torch.zeros(n_channels, dtype=torch.float64)
    std  = torch.zeros(n_channels, dtype=torch.float64)
    total_pixels = 0

    print(f"Calc stats for {len(file_list)} images with {n_channels} channels...")

    for images, _ in tqdm(loader):
        batch_size = images.size(0)
        images = images.double()

        images = images.view(batch_size, n_channels, -1)

        mean += images.mean(2).sum(0)
        std  += images.std(2).sum(0)
        total_pixels += batch_size

    mean /= total_pixels
    std  /= total_pixels

    return mean.tolist(), std.tolist()

## Data Load

In [9]:
class CoffeeDataset(Dataset):
    def __init__(self, file_list, augment=None, has_vis=HAS_VIS, vis_clip=VIS_CLIP_RANGE):
        self.file_list = file_list
        self.augment   = augment
        self.has_vis   = has_vis
        self.vis_clip = vis_clip

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        mask_dir = img_path.parent.parent / 'masks'
        mask_path = mask_dir / f"{img_path.stem}_mask.tif"

        with rasterio.open(img_path) as src:
            raw_data = src.read().astype('float32')

        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype('float32')

        raw_bands = raw_data[0:4, :, :]
        spectral_bands = raw_bands[[2, 1, 0, 3], :, :]

        #scale
        spectral_bands = spectral_bands / 10000.0
        spectral_bands = np.clip(spectral_bands, 0.0, 1.0)

        if self.has_vis:
            vis_band = raw_data[4:, :, :]
            vis_band = np.clip(vis_band, self.vis_clip[0], self.vis_clip[1])
            image = np.concatenate([spectral_bands, vis_band], axis=0)
        else:
            image = spectral_bands

        image = np.nan_to_num(image, nan=0.0)
        mask  = np.nan_to_num(mask, nan=0.0)

        image = image.transpose(1, 2, 0)

        if self.augment:
            aug   = self.augment(image=image, mask=mask)
            image = aug['image']
            mask  = aug['mask']

            if isinstance(image, np.ndarray):
                image = torch.from_numpy(image.transpose(2, 0, 1))
                mask  = torch.from_numpy(mask)
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1))
            mask  = torch.from_numpy(mask)

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        return image.float(), mask.float()

In [10]:
#train_imgs = list(DIRS['train']['img'].glob("*.tif"))

#calc_mean, calc_std = get_stats(train_imgs)

#print(f"DATASET_MEAN = {calc_mean}")
#print(f"DATASET_STD  = {calc_std}")


In [11]:
train_transforms = [
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=5,
        border_mode=cv2.BORDER_REFLECT,
        p=0.4
    )
]

val_transforms = []

if USE_NORMALIZATION:
    norm_op = A.Normalize(mean=DATASET_MEAN, std=DATASET_STD, max_pixel_value=1.0)
    train_transforms.append(norm_op)
    val_transforms.append(norm_op)

train_transforms.append(ToTensorV2())
val_transforms.append(ToTensorV2())

train_aug = A.Compose(train_transforms)
val_aug   = A.Compose(val_transforms)

train_files = sorted(list(DIRS['train']['img'].glob('*.tif')))
val_files   = sorted(list(DIRS['val']['img'].glob('*.tif')))
test_files  = sorted(list(DIRS['test']['img'].glob('*.tif')))

train_ds = CoffeeDataset(train_files, augment=train_aug, has_vis=HAS_VIS)
val_ds   = CoffeeDataset(val_files,   augment=val_aug,   has_vis=HAS_VIS)
test_ds  = CoffeeDataset(test_files,  augment=val_aug,   has_vis=HAS_VIS)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

total_samples = len(train_files) + len(val_files) + len(test_files)

print("=== Dataset distribution ===")
for name, files in [("Train", train_files), ("Val", val_files), ("Test", test_files)]:
    count = len(files)
    pct = (count / total_samples) * 100 if total_samples > 0 else 0
    print(f"{name:<5}: {count:4d} samples ({pct:.1f}%)")
print(f"Total: {total_samples}")

=== Dataset distribution ===
Train:  542 samples (60.1%)
Val  :  165 samples (18.3%)
Test :  195 samples (21.6%)
Total: 902


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


## Fine Tuning

### Archs

In [12]:
architectures = {
    "unetpp_resnet34": dict(
        cls=smp.UnetPlusPlus,
        kwargs=dict(
            encoder_name="resnet34",
            decoder_attention_type=None
        )
    ),

    "segformer_mit_b2": dict(
        cls=smp.Segformer,
        kwargs=dict(
            encoder_name="mit_b2"
        )
    ),

    "manet_resnet34": dict(
        cls=smp.MAnet,
        kwargs=dict(
            encoder_name="resnet34",
            decoder_pab_channels=64
        )
    ),
}

### Overview

In [13]:
print("_"*50)
print("             GENERAL INFORMATION")
print(f"• Train samples      : {len(train_ds)}")
print(f"• Validation samples : {len(val_ds)}")
print(f"• Test samples       : {len(test_ds)}")
print()
print(f"• Batch size         : {BATCH_SIZE}")
print(f"• Max Epochs         : {NUM_EPOCHS}")
print(f"• Early Stop Patience: {EARLY_STOP_PATIENCE}")
print(f"• Learning rate      : {LR:.1e}")
print(f"• Decision Threshold : {THR}")

print("\n• Selected Architectures:")
for nome, cfg in architectures.items():
    model_type = cfg['cls'].__name__
    params = ", ".join([f"{k}={v}" for k, v in cfg['kwargs'].items()])
    print(f"  – {nome} ({model_type})")
    print(f"    Config: {params}")

batch_imgs, batch_masks = next(iter(train_loader))

print(f"\n• Tensor Shape : {batch_imgs.shape}  [B, C, H, W]")
print(f"• Data Type    : {batch_imgs.dtype}")
print(f"• Min Value    : {batch_imgs.min().item():.3f}")
print(f"• Max Value    : {batch_imgs.max().item():.3f}")

print(f"\n• Tensor Shape (mask): {batch_masks.shape}")
print(f"• Data Type (mask)   : {batch_masks.dtype}")
print(f"• Min Value (mask)   : {batch_masks.min().item():.3f}")
print(f"• Max Value (mask)   : {batch_masks.max().item():.3f}")

unique_vals = torch.unique(batch_masks)
print(f"• Unique values      : {unique_vals.tolist()}")

if torch.isnan(batch_imgs).any() or torch.isinf(batch_imgs).any():
    print("\nWarning: Some samples has NaNs/Infs values.")
else:
    print("\n>> Data Integrity Check: OK")
print("_"*50)

interact(
    view_sample,
    index=IntSlider(min=0, max=BATCH_SIZE-1, step=1, value=0, description='Sample:')
);

__________________________________________________
             GENERAL INFORMATION
• Train samples      : 542
• Validation samples : 165
• Test samples       : 195

• Batch size         : 16
• Max Epochs         : 100
• Early Stop Patience: 15
• Learning rate      : 1.0e-04
• Decision Threshold : 0.5

• Selected Architectures:
  – unetpp_resnet34 (UnetPlusPlus)
    Config: encoder_name=resnet34, decoder_attention_type=None
  – segformer_mit_b2 (Segformer)
    Config: encoder_name=mit_b2
  – manet_resnet34 (MAnet)
    Config: encoder_name=resnet34, decoder_pab_channels=64

• Tensor Shape : torch.Size([16, 5, 256, 256])  [B, C, H, W]
• Data Type    : torch.float32
• Min Value    : -0.647
• Max Value    : 0.803

• Tensor Shape (mask): torch.Size([16, 1, 256, 256])
• Data Type (mask)   : torch.float32
• Min Value (mask)   : 0.000
• Max Value (mask)   : 1.000
• Unique values      : [0.0, 1.0]

>> Data Integrity Check: OK
__________________________________________________


interactive(children=(IntSlider(value=0, description='Sample:', max=15), Output()), _dom_classes=('widget-inte…

### Loss and Metrics

In [14]:
loss_dice = smp.losses.DiceLoss(
    mode="binary", from_logits=True, smooth=1., eps=1e-7
)

loss_tversky = smp.losses.TverskyLoss(
    mode="binary", from_logits=True, alpha=0.3, beta=0.7, smooth=1., eps=1e-7
)

loss_focal = smp.losses.FocalLoss(
    mode="binary", alpha=0.25, gamma=2.0
)

def dice_focal_loss(pred, target):
    return loss_dice(pred, target) + loss_focal(pred, target)

losses_to_test = {
    "Dice": loss_dice,
    #"Tversky": loss_tversky,
    #"DiceFocal": dice_focal_loss
}

metric_iou  = torchmetrics.JaccardIndex(task="binary", threshold=THR).to(DEVICE)
metric_f1   = torchmetrics.F1Score(task="binary", threshold=THR).to(DEVICE)
metric_acc  = torchmetrics.Accuracy(task="binary", threshold=THR).to(DEVICE)
metric_rec  = torchmetrics.Recall(task="binary", threshold=THR).to(DEVICE)
metric_prec = torchmetrics.Precision(task="binary", threshold=THR).to(DEVICE)

### Train, Valid and Test

In [15]:
all_models_stats = []

for loss_name, criterion in losses_to_test.items():

    print(f"\n{'_'*60}")
    print(f"STARTING EXPERIMENT WITH LOSS: {loss_name}")

    for model_name, cfg in architectures.items():

        experiment_id = f"{model_name}_{loss_name}"
        print(f"\n>> Training {model_name} with {loss_name}...\n")

        model = cfg["cls"](
            **cfg["kwargs"],
            encoder_weights="imagenet",
            in_channels=SAMPLE_CHANNELS,
            classes=1,
            activation=None
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3
        )

        writer = SummaryWriter(log_dir=f"{BASE_DIR}/runs/{experiment_id}")

        best_metric = 0.0
        epochs_no_improve = 0

        for epoch in range(1, NUM_EPOCHS + 1):

            t0 = time.time()

            #train
            model.train()
            train_loss_accum = 0.0

            for imgs, masks in train_loader:

                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

                optimizer.zero_grad(set_to_none=True)

                logits = model(imgs)
                loss_val = criterion(logits, masks)

                loss_val.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                train_loss_accum += loss_val.item() * imgs.size(0)

            train_loss = train_loss_accum / len(train_loader.dataset)

            #validation
            model.eval()
            val_loss_accum = 0.0

            metric_iou.reset(); metric_f1.reset()
            metric_acc.reset(); metric_rec.reset(); metric_prec.reset()

            with torch.no_grad():
                for imgs, masks in val_loader:

                    imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

                    logits = model(imgs)
                    loss_val = criterion(logits, masks)

                    val_loss_accum += loss_val.item() * imgs.size(0)

                    preds = torch.sigmoid(logits)

                    metric_iou.update(preds, masks)
                    metric_f1.update(preds, masks)
                    metric_acc.update(preds, masks)
                    metric_rec.update(preds, masks)
                    metric_prec.update(preds, masks)

            val_loss = val_loss_accum / len(val_loader.dataset)

            final_val_iou  = metric_iou.compute().item()
            final_val_f1   = metric_f1.compute().item()
            final_val_acc  = metric_acc.compute().item()
            final_val_rec  = metric_rec.compute().item()
            final_val_prec = metric_prec.compute().item()

            scheduler.step(val_loss)

            #tensorBoard logs
            writer.add_scalar("Loss/train", train_loss, epoch)
            writer.add_scalar("Loss/val",   val_loss,   epoch)

            writer.add_scalar("IoU/val",    final_val_iou,  epoch)
            writer.add_scalar("F1/val",     final_val_f1,   epoch)
            writer.add_scalar("Acc/val",    final_val_acc,  epoch)
            writer.add_scalar("Recall/val", final_val_rec,  epoch)
            writer.add_scalar("Prec/val",   final_val_prec, epoch)

            print(
                f"[{loss_name}] {model_name} | Ep {epoch:02}/{NUM_EPOCHS} | "
                f"L_tr={train_loss:.4f} L_val={val_loss:.4f} | "
                f"IoU={final_val_iou:.4f} | F1={final_val_f1:.4f} | "
                f"Acc={final_val_acc:.4f} | Rec={final_val_rec:.4f} | Prec={final_val_prec:.4f} | "
                f"T={int(time.time()-t0)}s"
            )

            current_metric = final_val_iou
            if current_metric > best_metric:

                best_metric = current_metric
                epochs_no_improve = 0

                torch.save({
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "best_metric": best_metric,
                    "model_name": model_name,
                    "loss_name": loss_name
                }, f"{BASE_DIR}/best_{experiment_id}.pth")
                print(f"  >>> New Best IoU: {best_metric:.4f}")

            else:
                epochs_no_improve += 1

            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                break

            torch.cuda.empty_cache()
            gc.collect()

        print(f"Best Val IoU for {experiment_id}: {best_metric:.4f}")

        #test
        ckpt = torch.load(f"{BASE_DIR}/best_{experiment_id}.pth", map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        model.eval()

        metric_iou.reset(); metric_f1.reset()
        metric_acc.reset(); metric_rec.reset(); metric_prec.reset()

        raw_results = []

        with torch.no_grad():
            for batch_idx, (imgs, masks) in enumerate(test_loader):

                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

                logits = model(imgs)
                preds = torch.sigmoid(logits)
                pred_binary = (preds > THR).float()

                metric_iou.update(preds, masks)
                metric_f1.update(preds, masks)
                metric_acc.update(preds, masks)
                metric_rec.update(preds, masks)
                metric_prec.update(preds, masks)

                for i in range(imgs.size(0)):

                    global_idx = batch_idx * BATCH_SIZE + i
                    if global_idx >= len(test_files):
                        break

                    file_name = test_files[global_idx].name

                    p = pred_binary[i]
                    t = masks[i]

                    intersection = (p * t).sum().item()
                    union = p.sum().item() + t.sum().item() - intersection

                    sample_iou = (intersection + 1e-7) / (union + 1e-7)
                    sample_f1  = (2 * intersection + 1e-7) / (
                        p.sum().item() + t.sum().item() + 1e-7
                    )

                    raw_results.append({
                        "image_id": file_name,
                        "iou": sample_iou,
                        "f1": sample_f1,
                        "pixels_gt": t.sum().item(),
                        "pixels_pred": p.sum().item()
                    })

        df_results = pd.DataFrame(raw_results)
        df_results.to_csv(f"{BASE_DIR}/results_{experiment_id}.csv", index=False)

        test_iou  = metric_iou.compute().item()
        test_f1   = metric_f1.compute().item()
        test_acc  = metric_acc.compute().item()
        test_rec  = metric_rec.compute().item()
        test_prec = metric_prec.compute().item()

        writer.add_scalar("IoU/test", test_iou, NUM_EPOCHS)
        writer.add_scalar("F1/test",  test_f1,  NUM_EPOCHS)
        writer.add_scalar("Acc/test", test_acc, NUM_EPOCHS)
        writer.add_scalar("Recall/test", test_rec, NUM_EPOCHS)
        writer.add_scalar("Prec/test",   test_prec, NUM_EPOCHS)

        writer.close()

        del model
        torch.cuda.empty_cache()
        gc.collect()


____________________________________________________________
STARTING EXPERIMENT WITH LOSS: Dice

>> Training unetpp_resnet34 with Dice...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

[Dice] unetpp_resnet34 | Ep 01/100 | L_tr=0.7852 L_val=0.6908 | IoU=0.2160 | F1=0.3552 | Acc=0.5695 | Rec=0.9463 | Prec=0.2187 | T=12s
  >>> New Best IoU: 0.2160
[Dice] unetpp_resnet34 | Ep 02/100 | L_tr=0.6931 L_val=0.5573 | IoU=0.4542 | F1=0.6247 | Acc=0.8773 | Rec=0.8146 | Prec=0.5066 | T=10s
  >>> New Best IoU: 0.4542
[Dice] unetpp_resnet34 | Ep 03/100 | L_tr=0.6445 L_val=0.5199 | IoU=0.5056 | F1=0.6717 | Acc=0.9043 | Rec=0.7811 | Prec=0.5891 | T=10s
  >>> New Best IoU: 0.5056
[Dice] unetpp_resnet34 | Ep 04/100 | L_tr=0.5974 L_val=0.4851 | IoU=0.5255 | F1=0.6889 | Acc=0.9094 | Rec=0.8005 | Prec=0.6047 | T=10s
  >>> New Best IoU: 0.5255
[Dice] unetpp_resnet34 | Ep 05/100 | L_tr=0.5640 L_val=0.4896 | IoU=0.4750 | F1=0.6441 | Acc=0.8834 | Rec=0.8418 | Prec=0.5215 | T=10s
[Dice] unetpp_resnet34 | Ep 06/100 | L_tr=0.5235 L_val=0.4405 | IoU=0.5462 | F1=0.7065 | Acc=0.9167 | Rec=0.7996 | Prec=0.6328 | T=10s
  >>> New Best IoU: 0.5462
[Dice] unetpp_resnet34 | Ep 07/100 | L_tr=0.4859 L_val=

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

[Dice] segformer_mit_b2 | Ep 01/100 | L_tr=0.7218 L_val=0.5895 | IoU=0.3106 | F1=0.4740 | Acc=0.7567 | Rec=0.8746 | Prec=0.3251 | T=13s
  >>> New Best IoU: 0.3106
[Dice] segformer_mit_b2 | Ep 02/100 | L_tr=0.5573 L_val=0.4335 | IoU=0.4812 | F1=0.6498 | Acc=0.9024 | Rec=0.7222 | Prec=0.5905 | T=6s
  >>> New Best IoU: 0.4812
[Dice] segformer_mit_b2 | Ep 03/100 | L_tr=0.4855 L_val=0.4172 | IoU=0.4974 | F1=0.6644 | Acc=0.9068 | Rec=0.7361 | Prec=0.6054 | T=6s
  >>> New Best IoU: 0.4974
[Dice] segformer_mit_b2 | Ep 04/100 | L_tr=0.4387 L_val=0.4058 | IoU=0.4654 | F1=0.6352 | Acc=0.8797 | Rec=0.8357 | Prec=0.5123 | T=6s
[Dice] segformer_mit_b2 | Ep 05/100 | L_tr=0.4116 L_val=0.3840 | IoU=0.5008 | F1=0.6674 | Acc=0.9248 | Rec=0.6022 | Prec=0.7484 | T=6s
  >>> New Best IoU: 0.5008
[Dice] segformer_mit_b2 | Ep 06/100 | L_tr=0.3912 L_val=0.3406 | IoU=0.5401 | F1=0.7014 | Acc=0.9228 | Rec=0.7232 | Prec=0.6809 | T=6s
  >>> New Best IoU: 0.5401
[Dice] segformer_mit_b2 | Ep 07/100 | L_tr=0.3681 L_va

## Export Results

In [16]:
ls

drive/  input/  sample_data/  working_10_04_2026_03_34_41/


In [17]:
!zip -r results_{TIMESTAMP}.zip {BASE_DIR}

  adding: content/working_10_04_2026_03_34_41/ (stored 0%)
  adding: content/working_10_04_2026_03_34_41/results_segformer_mit_b2_Dice.csv (deflated 70%)
  adding: content/working_10_04_2026_03_34_41/runs/ (stored 0%)
  adding: content/working_10_04_2026_03_34_41/runs/unetpp_resnet34_Dice/ (stored 0%)
  adding: content/working_10_04_2026_03_34_41/runs/unetpp_resnet34_Dice/events.out.tfevents.1775792088.9ed8851542c1.27232.0 (deflated 66%)
  adding: content/working_10_04_2026_03_34_41/runs/manet_resnet34_Dice/ (stored 0%)
  adding: content/working_10_04_2026_03_34_41/runs/manet_resnet34_Dice/events.out.tfevents.1775793169.9ed8851542c1.27232.2 (deflated 66%)
  adding: content/working_10_04_2026_03_34_41/runs/segformer_mit_b2_Dice/ (stored 0%)
  adding: content/working_10_04_2026_03_34_41/runs/segformer_mit_b2_Dice/events.out.tfevents.1775792780.9ed8851542c1.27232.1 (deflated 65%)
  adding: content/working_10_04_2026_03_34_41/best_manet_resnet34_Dice.pth (deflated 9%)
  adding: content/wor

In [18]:
!cp results_{TIMESTAMP}.zip /content/drive/MyDrive/Datasets/results/s2/